# Football: Sports Analytics

**Sports Analytics Club at UT Dallas · Workshop 2**

**Edit this cell** marks your tasks. **Run this cell** means the code is ready; leave it as provided.
Work from top to bottom and replace `____` using the example.

> **Sport in 30 seconds**
> - One row = one NFL game.
> - Home and away are the two teams playing.
> - We predict the home team’s final points (`home_score`).
> - Points allowed = points a team gave up.
> - Rest = days since the previous game.

## Setup

Connect to a Colab **CPU** runtime and run both collapsed cells. The dataset is included and its checksum is checked.

In [ ]:
#@title Import libraries — run this cell
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

SEED = 42
plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('pandas:', pd.__version__, '| scikit-learn:', sklearn.__version__)


In [ ]:
#@title Prepared dataset — run this cell (expand only if curious)
# Prepared data snapshot. No network calls. Instructor rebuilds with scripts/prepare_data.py.
import base64, hashlib, io, zipfile
EXPECTED_SHA256 = {'nfl_data.csv': 'b023e08a7cea95355a70313f1d1ace50f197c51a54181e2e49532815648c0ddd'}
SNAPSHOT_B64 = "..."
candidates = [Path('data'), Path('../data')]
DATA_DIR = next((p for p in candidates if all((p/n).is_file() for n in EXPECTED_SHA256)), None)
if DATA_DIR is None:
    DATA_DIR = Path('workshop2_data')
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(SNAPSHOT_B64))) as archive:
        for name in EXPECTED_SHA256:
            (DATA_DIR/name).write_bytes(archive.read(name))
for name, expected in EXPECTED_SHA256.items():
    actual = hashlib.sha256((DATA_DIR/name).read_bytes()).hexdigest()
    assert actual == expected, f'{name} differs from the workshop snapshot. Restore the supplied CSV.'
print('Ready: 1 verified CSV.')


## First look

Run this cell to preview the data.

In [ ]:
football = pd.read_csv(DATA_DIR/'nfl_data.csv')
for name, frame in [('NFL', football)]:
    print(name, 'rows/columns:', frame.shape, '| missing cells:', frame.isna().sum().sum())
    display(frame.head(3))


| Column | Plain English |
|---|---|
| `home_score` | Home team’s final points, including overtime — our target |
| `home_prior4_scored` | Home team’s average points scored over its previous 4 games |
| `away_prior4_allowed` | Away team’s average points given up over its previous 4 games |
| `home_rest`, `away_rest` | Days since each team’s previous game |

## Predict home points

Train on **2017–2022**; test on **2023–2024**. Inputs use games played before kickoff.

**Predict first:** Do you think recent scoring history will beat simply predicting the average score?

In [ ]:
#@title Run this cell — prepare training and test games
# Prepared split: earlier seasons teach the model; later seasons test it.
reg_train = football.loc[football.season <= 2022].copy()
reg_test = football.loc[football.season >= 2023].copy()
y_train = reg_train['home_score']
y_test = reg_test['home_score']
print('Train games:', len(reg_train), '| Test games:', len(reg_test))


### Worked example — run this cell

In [ ]:
# Worked example: select pregame inputs, fit, then predict three test scores.
example_features = ['home_prior4_scored', 'away_prior4_allowed']
example_regressor = LinearRegression()
example_regressor.fit(reg_train[example_features], y_train)
print(example_regressor.predict(reg_test[example_features].iloc[:3]))

### F1 · Edit this cell

Use the same two features and model as the example. Train with `reg_train[reg_features]` and `y_train`.

In [ ]:
# Choose the pregame features
reg_features = ____

# Create the model
reg_model = ____

# Train it
reg_model.fit(____, ____)


In [ ]:
#@title Run this cell — show MAE, predictions, and coefficients
# Check the selected inputs before reporting test results.
assert reg_features and set(reg_features) <= {'home_prior4_scored', 'away_prior4_allowed', 'home_rest', 'away_rest'}
X_train = reg_train[reg_features]
X_test = reg_test[reg_features]
print('Features:', reg_features, '| Target: home_score')
# Linear Regression estimates averages; predictions can be fractional or negative.
reg_predictions = reg_model.predict(X_test)
baseline_predictions = np.full(len(y_test), y_train.mean())
reg_results = pd.DataFrame({
    'model': ['Training-mean baseline', 'Linear Regression'],
    'test_MAE_points': [mean_absolute_error(y_test, baseline_predictions),
                        mean_absolute_error(y_test, reg_predictions)]
})
display(reg_results.round(3))

fig, ax = plt.subplots()
ax.scatter(y_test, reg_predictions, alpha=0.35, color='#157F78')
limit = max(y_test.max(), reg_predictions.max()) + 3
ax.plot([0, limit], [0, limit], '--', color='gray', label='Perfect predictions')
ax.set(xlabel='Actual home points', ylabel='Predicted home points',
       title='NFL: predictions on 2023–2024 games')
ax.legend(); plt.tight_layout(); plt.show()
# Coefficients describe associations, not causes.
display(pd.Series(reg_model.coef_, index=reg_features, name='Coefficient').round(3))


**MAE** is the average prediction error in points; lower is better.

A positive coefficient means higher values of that feature are associated with a higher predicted score, holding the other selected features fixed.

### F2 · Try one change

Add `home_rest` and `away_rest` to F1’s feature list. Run F1, then the next **Run this cell** output.

**Did MAE improve meaningfully? Report the change in points.**

This compares the same test games; a future-season check would be needed before trusting the improvement.

<details>
<summary>Sources and definitions</summary>

Prepared files and complete column definitions: `data/DATA_SOURCES.md` and `data/DATA_DICTIONARY.md` in the package.
Raw-source URLs, checksums, and row-filter audit: `data/manifest.json`.

- **NFL:** [nflverse nfldata schedules](https://github.com/nflverse/nfldata/blob/25cd9bb9587f693ef78ecb05e8b2ba1138b86f5d/data/games.csv), [field dictionary](https://nflreadr.nflverse.com/articles/dictionary_schedules.html).

- **Methods:** [Linear Regression](https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares) · [MAE](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html)

</details>